<a href="https://colab.research.google.com/github/jhon966/EX2_C-/blob/main/EX1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

!pip install transformers torch

In [ ]:
import torch
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch.nn.functional as F
from collections import defaultdict



# 1. Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Load Model and Tokenizer
# Note: Changed from 'gpt2-large' to 'gpt2' (small) to match the standard
# 12 layers x 12 heads architecture (144 heads total) typically used in this assignment.
model_name = "gpt2-large"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.eval()

# GPT-2 doesn't have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

print(f"{model_name} is ready on {device}!")

Using device: cuda


Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


gpt2-large is ready on cuda!


In [ ]:
def get_subject_last_token_idx(prompt, subject, tokenizer):
    """
    Finds the token index of the last part of the subject within the prompt.
    Uses simple string matching to find the prefix up to the end of the subject.
    """
    # Find where the subject ends in the prompt string
    subject_end_idx = prompt.find(subject) + len(subject)

    # Extract the string up to the end of the subject
    prefix = prompt[:subject_end_idx]

    # Tokenize the prefix. The last token of this sequence is our target index.
    prefix_tokens = tokenizer.encode(prefix)

    return len(prefix_tokens) - 1

def get_ablation_hook(head_indices):
    """
    Creates a PyTorch forward hook that zeroes out specific attention heads.
    head_indices: list of head indices (0-11) to ablate in a specific layer.
    """
    def hook(module, inputs, output):
        # output[0] is the attention output. Shape: (batch, seq_len, hidden_size)
        attn_output = output[0]
        head_dim = module.head_dim # Typically 64 for gpt2-small

        # Zero out the specific slice of the hidden dimension for each head
        for head_idx in head_indices:
            start_idx = head_idx * head_dim
            end_idx = (head_idx + 1) * head_dim
            attn_output[:, :, start_idx:end_idx] = 0.0

        # Return the modified tuple to continue the forward pass
        return (attn_output,) + output[1:]

    return hook

In [ ]:

# 1. Extract the file ID from your link
file_id = '1IcPzaOqa23bAhjz1rHiKRwZCokvHOICl'

# 2. Construct the direct download URL for Google Drive
direct_link = f'https://drive.google.com/uc?export=download&id={file_id}'

# 3. Read the CSV directly from the web
df = pd.read_csv(direct_link)

print(f"Successfully loaded {len(df)} rows from Google Drive!")

# Define column names based on your CSV structure
prompt_col = 'Prompt'
subject_col = 'Subject Word(s)'
target_col = 'Target Token'

# Initialize new columns for the results
df['max_head_layer'] = -1
df['max_head_index'] = -1
df['condition_a_delta'] = 0.0
df['condition_b_delta'] = 0.0
df['condition_c_delta'] = 0.0

print(f"Starting analysis for {len(df)} rows...")

# 2. Main Loop
for idx, row in df.iterrows():
    prompt = row[prompt_col]
    subject = row[subject_col]
    # Ensure target is a string and add a leading space (GPT-2 standard behavior)
    target = " " + str(row[target_col]).strip()

    # Prepare inputs
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    target_token_id = tokenizer.encode(target)[0]
    subj_idx = get_subject_last_token_idx(prompt, subject, tokenizer)

    # ==========================================
    # STEP A: BASELINE RUN & HEAD IDENTIFICATION
    # ==========================================
    with torch.no_grad():
        # output_attentions=True is crucial here
        outputs = model(**inputs, output_attentions=True)

    # Calculate baseline probability of the target token
    logits = outputs.logits[0, -1, :] # Logits of the very last token
    probs = F.softmax(logits, dim=-1)
    p_baseline = probs[target_token_id].item()

    # Analyze Attention Weights
    # Attentions structure: Tuple of 12 layers -> (batch, num_heads, seq_len, seq_len)
    attentions = outputs.attentions

    head_scores = []
    for layer_idx in range(model.config.n_layer):
        for head_idx in range(model.config.n_head):
            # Attention from the last token (-1) to the subject token (subj_idx)
            weight = attentions[layer_idx][0, head_idx, -1, subj_idx].item()
            head_scores.append((layer_idx, head_idx, weight))

    # Sort heads by attention weight (highest first)
    head_scores.sort(key=lambda x: x[2], reverse=True)

    # Define the 3 conditions
    cond_a_heads = [head_scores[0]]                     # Condition A: Top 1 head
    cond_b_heads = head_scores[:3]                      # Condition B: Top 3 heads
    cond_c_heads = [h for h in head_scores if h[2] > 0.25] # Condition C: > 25% attention

    # Document the top head for the final submission
    df.at[idx, 'max_head_layer'] = cond_a_heads[0][0]
    df.at[idx, 'max_head_index'] = cond_a_heads[0][1]

    # ==========================================
    # STEP B: INTERVENTION RUNS (ABLATION)
    # ==========================================
    def run_intervention(heads_to_ablate):
        # If no heads meet the criteria (e.g., Condition C), probability doesn't change
        if not heads_to_ablate:
            return p_baseline

        # Group heads by layer so we only attach one hook per layer
        layer_to_heads = defaultdict(list)
        for l, h, _ in heads_to_ablate:
            layer_to_heads[l].append(h)

        handles = []
        # Register hooks dynamically
        for layer_idx, h_list in layer_to_heads.items():
            module = model.transformer.h[layer_idx].attn
            handle = module.register_forward_hook(get_ablation_hook(h_list))
            handles.append(handle)

        # Forward pass with ablations
        with torch.no_grad():
            int_outputs = model(**inputs)

        # IMPORTANT: Remove hooks immediately to clean up for the next run
        for handle in handles:
            handle.remove()

        # Calculate new probability
        int_logits = int_outputs.logits[0, -1, :]
        int_probs = F.softmax(int_logits, dim=-1)
        return int_probs[target_token_id].item()

    # Execute interventions and calculate deltas
    # Formula: (P_baseline - P_intervention) / P_baseline

    if p_baseline > 0: # Prevent division by zero
        p_a = run_intervention(cond_a_heads)
        df.at[idx, 'condition_a_delta'] = (p_baseline - p_a) / p_baseline

        p_b = run_intervention(cond_b_heads)
        df.at[idx, 'condition_b_delta'] = (p_baseline - p_b) / p_baseline

        p_c = run_intervention(cond_c_heads)
        df.at[idx, 'condition_c_delta'] = (p_baseline - p_c) / p_baseline

    # Print progress every 50 rows
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1} rows...")

# 3. Save Final Results
output_filename = 'ID_results.csv' # Rename 'ID' to your actual student ID
df.to_csv(output_filename, index=False)
print(f"Done! Results saved to {output_filename}")

Successfully loaded 80 rows from Google Drive!
Starting analysis for 80 rows...


TypeError: 'NoneType' object is not subscriptable